In [1]:
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs])

pip_install([
    "feast==0.45.*",
    "pandas>=2.2.0",
    "pyarrow>=15.0.0",
    "numpy>=1.26.0",
    "google-cloud-storage>=2.14.0",
    # optional for git commits from notebook (later cell)
    "GitPython>=3.1.41",
])
print("✅ Packages installed")


  Using cached pyarrow-18.0.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (3.3 kB)
Using cached pyarrow-18.0.0-cp310-cp310-manylinux_2_28_x86_64.whl (40.0 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
✅ Packages installed


In [2]:
import sys, subprocess

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", *args])

pip("install", "--upgrade", "pip")
pip("install", "feast==0.45.*", "pandas>=2.2.0,<2.3.0", "numpy>=1.26.0", "pyarrow>=14.0.2", "fastparquet==2024.5.0")
print("✅ Packages ready (using fastparquet for writes)")

✅ Packages ready (using fastparquet for writes)


In [10]:
import sys, subprocess
def pip(*args): subprocess.check_call([sys.executable, "-m", "pip", *args])

pip("install", "--upgrade", "pip")
pip("install", "feast==0.45.*", "pandas>=2.2.0,<2.3.0", "numpy>=1.26.0", "pyarrow==14.0.2", "fastparquet==2024.5.0")
print("✅ Packages installed/aligned")


✅ Packages installed/aligned


In [4]:
from pathlib import Path
import os, yaml

# 1) Make sure repo folder exists
repo = Path("feast_repo")
repo.mkdir(parents=True, exist_ok=True)

# 2) Write a clean feature_store.yaml that keeps registry/online DB in feast_repo/
fixed_cfg = {
    "project": "stock_analytica",
    "provider": "local",
    "entity_key_serialization_version": 2,
    "registry": {"path": "registry.db"},
    "online_store": {"type": "sqlite", "path": "online_store.db"},
    "offline_store": {"type": "file"},
}
with open(repo / "feature_store.yaml", "w") as f:
    yaml.safe_dump(fixed_cfg, f, sort_keys=False)

# 3) ALSO pre-create the *old* path in case Feast still tries it this run
#    (avoids FileNotFoundError due to mkdir(parents=False) in Feast)
old_dir = repo / "data" / "feast"
old_dir.mkdir(parents=True, exist_ok=True)

# 4) Show what Feast will read now
print((repo / "feature_store.yaml").read_text())
print("Precreated fallback dir:", old_dir.as_posix())


project: stock_analytica
provider: local
entity_key_serialization_version: 2
registry:
  path: registry.db
online_store:
  type: sqlite
  path: online_store.db
offline_store:
  type: file

Precreated fallback dir: feast_repo/data/feast


In [11]:
from google.cloud import storage
from pathlib import Path

BUCKET = "mlopsga1_data"
PREFIX_V0 = "oppe1/StockAnalyticaData/v0/"
PREFIX_V1 = "oppe1/StockAnalyticaData/v1/"

client = storage.Client()
bucket = client.bucket(BUCKET)

def download_prefix(prefix, local_dir):
    Path(local_dir).mkdir(parents=True, exist_ok=True)
    n = 0
    for blob in client.list_blobs(BUCKET, prefix=prefix):
        if blob.name.endswith("/") or not blob.name.lower().endswith(".csv"): 
            continue
        rel = blob.name[len(prefix):]
        dest = Path(local_dir) / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        blob.download_to_filename(dest.as_posix())
        n += 1
    return n

n0 = download_prefix(PREFIX_V0, "data/StockAnalyticaData/v0")
n1 = download_prefix(PREFIX_V1, "data/StockAnalyticaData/v1")
print(f"✅ Downloaded v0: {n0}, v1: {n1}")


✅ Downloaded v0: 2, v1: 3


In [12]:
import pandas as pd, glob, os
from pathlib import Path

def load_all_csvs(local_dir: str) -> pd.DataFrame:
    dfs = []
    for f in glob.glob(os.path.join(local_dir, "*.csv")):
        df = pd.read_csv(f)
        if "timestamp" not in df.columns:
            raise ValueError(f"{f} missing 'timestamp'")
        stock_name = Path(f).name.split("__")[0]
        df["stock_name"] = stock_name
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def compute_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values(["stock_name", "timestamp"]).reset_index(drop=True)

    def _feat(g):
        g = g.set_index("timestamp")
        g["rolling_avg_10"] = g["close"].rolling("10min", min_periods=1).mean()
        g["volume_sum_10"] = g["volume"].rolling("10min", min_periods=1).sum()
        return g.reset_index()

    parts = []
    for stock, g in df.groupby("stock_name", sort=False):
        base = g[["timestamp", "close", "volume"]]
        rolled = _feat(base)
        rolled["stock_name"] = stock
        parts.append(rolled)

    out = pd.concat(parts, ignore_index=True)
    out = out.rename(columns={"timestamp": "event_timestamp"})
    return out[["stock_name", "event_timestamp", "rolling_avg_10", "volume_sum_10"]].dropna()

def save_parquet_per_stock(df: pd.DataFrame, out_dir: str):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    for stock, g in df.groupby("stock_name", sort=False):
        g.to_parquet(Path(out_dir, f"{stock}.parquet"), index=False, engine="fastparquet")

# v0
v0_dir = "data/StockAnalyticaData/v0"
v1_dir = "data/StockAnalyticaData/v1"

v0_raw = load_all_csvs(v0_dir)
if v0_raw.empty: 
    raise SystemExit("❌ v0 raw empty. Ensure CSVs in data/StockAnalyticaData/v0")
v0_feat = compute_features(v0_raw)
save_parquet_per_stock(v0_feat, "data/features/v0")
print("✅ v0 features → data/features/v0")

# v1 (if present)
v1_raw = load_all_csvs(v1_dir)
if not v1_raw.empty:
    v1_feat = compute_features(v1_raw)
    save_parquet_per_stock(v1_feat, "data/features/v1")
    print("✅ v1 features → data/features/v1")
else:
    print("⚠️ v1 raw not found/empty; continuing with v0 only.")

# merged current
pieces = []
for folder in ["data/features/v0", "data/features/v1"]:
    for f in glob.glob(os.path.join(folder, "*.parquet")):
        pieces.append(pd.read_parquet(f, engine="fastparquet"))
if not pieces:
    raise SystemExit("❌ No feature files found to merge.")
cur = pd.concat(pieces, ignore_index=True)
save_parquet_per_stock(cur, "data/features/current")
print("✅ merged features → data/features/current")


✅ v0 features → data/features/v0
✅ v1 features → data/features/v1
✅ merged features → data/features/current


In [16]:
# Cell 1 — write a clean YAML file using PyYAML (no quotes or \n)
import yaml
from pathlib import Path

repo = Path("feast_repo")
repo.mkdir(parents=True, exist_ok=True)

cfg = {
    "project": "stock_analytica",
    "provider": "local",
    "entity_key_serialization_version": 2,
    "registry": {"path": "registry.db"},
    "online_store": {"type": "sqlite", "path": "online_store.db"},
    "offline_store": {"type": "file"},
}

yaml_path = repo / "feature_store.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

# Validate it parses and show the contents
with open(yaml_path, "r", encoding="utf-8") as f:
    parsed = yaml.safe_load(f)
print("Parsed OK:", parsed)
print("\n--- feature_store.yaml ---\n", yaml_path.read_text())


Parsed OK: {'project': 'stock_analytica', 'provider': 'local', 'entity_key_serialization_version': 2, 'registry': {'path': 'registry.db'}, 'online_store': {'type': 'sqlite', 'path': 'online_store.db'}, 'offline_store': {'type': 'file'}}

--- feature_store.yaml ---
 project: stock_analytica
provider: local
entity_key_serialization_version: 2
registry:
  path: registry.db
online_store:
  type: sqlite
  path: online_store.db
offline_store:
  type: file



In [18]:
from datetime import datetime, timezone
from pathlib import Path
from feast import FeatureStore, Entity, Field, FeatureView, FileSource
from feast.types import Float32, ValueType
from feast.data_format import ParquetFormat

# Absolute path to features
features_abs = (Path.cwd() / "data" / "features" / "current").resolve().as_posix()

file_source = FileSource(
    path=features_abs,
    event_timestamp_column="event_timestamp",
    file_format=ParquetFormat(),
)

stock = Entity(
    name="stock_name",
    join_keys=["stock_name"],
    description="Stock ticker/name",
    value_type=ValueType.STRING,   # deprecation-safe
)

stock_minute_features = FeatureView(
    name="stock_minute_features",
    entities=[stock],
    schema=[
        Field(name="rolling_avg_10", dtype=Float32),
        Field(name="volume_sum_10", dtype=Float32),
    ],
    source=file_source,
    online=True,
)

store = FeatureStore(repo_path=str((Path.cwd() / "feast_repo").resolve()))
store.apply([file_source, stock, stock_minute_features])
print("✅ Feast apply complete")

# CORRECT ARGUMENT NAMES:
start = datetime(2000, 1, 1, tzinfo=timezone.utc)
end = datetime.now(timezone.utc)
store.materialize(start_date=start, end_date=end)  # <-- use start_date / end_date
print("✅ Materialized to:", (Path(store.repo_path) / store.config.online_store.path))


✅ Feast apply complete
Materializing 1 feature views from 2000-01-01 00:00:00+00:00 to 2025-11-09 13:13:17+00:00 into the sqlite online store.

stock_minute_features:


100%|████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 235.01it/s]

✅ Materialized to: /home/jupyter/feast_repo/online_store.db


In [19]:
import pandas as pd, glob, os

files = glob.glob("data/features/current/*.parquet")
stocks_available = [os.path.basename(f).split(".")[0] for f in files]
print("Stocks available:", stocks_available[:10], "… total:", len(stocks_available))

entity_rows = [{"stock_name": s} for s in stocks_available[:5]]
features = [
    "stock_minute_features:rolling_avg_10",
    "stock_minute_features:volume_sum_10",
]
res = store.get_online_features(features=features, entity_rows=entity_rows).to_dict()
pd.DataFrame(res)


Stocks available: ['ADANIENT', 'ADANIGAS', 'ABFRL', 'AARTIIND', 'ABCAPITAL'] … total: 5


/opt/conda/lib/python3.10/site-packages/feast/entity.py:173: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'stock_name'.
  entity = cls(
/opt/conda/lib/python3.10/site-packages/feast/entity.py:173: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity '__dummy'.
  entity = cls(


,stock_name,volume_sum_10,rolling_avg_10
0,ADANIENT,195153.0,489.500000
1,ADANIGAS,64280.0,376.700012
2,ABFRL,72398.0,166.494995
3,AARTIIND,15301.0,1260.199951
4,ABCAPITAL,331896.0,88.675003
